# Notebook: 02_feature_engineering.ipynb

## Purpose
Clean and normalize Transfermarkt national-team market value data.

This notebook prepares team-year market value features that can later be joined into the final match-level modeling dataset.

## Input
`data/raw/transfermarkt/transfermarkt_elo_teams_squad_values_2004_2026.csv`

## Output
`data/processed/transfermarkt_market_values_clean.csv`

## Notes
- Transfermarkt data starts in 2004.
- Values are normalized by year to reduce football market inflation effects.
- This notebook does not train models.

## Imports + Paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 120)

current = Path.cwd()

if (current / "data").exists():
    BASE_DIR = current
elif (current.parent / "data").exists():
    BASE_DIR = current.parent
else:
    raise FileNotFoundError("Could not find project root with data/ folder")

RAW_TM_DIR = BASE_DIR / "data" / "raw" / "transfermarkt"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_FILE = RAW_TM_DIR / "transfermarkt_elo_teams_squad_values_2004_2026.csv"
OUT_FILE = PROCESSED_DIR / "transfermarkt_market_values_clean.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_FILE:", RAW_FILE)
print("OUT_FILE:", OUT_FILE)

RAW_FILE: /workspaces/World_cup_predictor/data/raw/transfermarkt/transfermarkt_elo_teams_squad_values_2004_2026.csv
OUT_FILE: /workspaces/World_cup_predictor/data/processed/transfermarkt_market_values_clean.csv


## Load Raw Transfermarkt Data

In [2]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f"Missing raw Transfermarkt file: {RAW_FILE}")

tm_raw = pd.read_csv(RAW_FILE)

print("Raw Transfermarkt shape:", tm_raw.shape)
display(tm_raw.head())
display(tm_raw.dtypes)

Raw Transfermarkt shape: (5014, 9)


,team_name_tm,team_slug_tm,team_id_tm,season_id,selected_year,squad_size_with_values,squad_market_value_millions_eur,avg_player_value_millions_eur,url
0,France,frankreich,3377,2004,2004.0,47,466.65,9.93,https://www.transfermarkt.com/frankreich/kader...
1,France,frankreich,3377,2005,2005.0,32,333.75,10.43,https://www.transfermarkt.com/frankreich/kader...
2,France,frankreich,3377,2006,2006.0,33,350.90,10.63,https://www.transfermarkt.com/frankreich/kader...
3,France,frankreich,3377,2007,2007.0,36,357.20,9.92,https://www.transfermarkt.com/frankreich/kader...
4,France,frankreich,3377,2008,2008.0,38,544.00,14.32,https://www.transfermarkt.com/frankreich/kader...


team_name_tm                           str
team_slug_tm                           str
team_id_tm                           int64
season_id                            int64
selected_year                      float64
squad_size_with_values               int64
squad_market_value_millions_eur    float64
avg_player_value_millions_eur      float64
url                                    str
dtype: object

##  Basic Validation + Cleaning

In [3]:
required_cols = [
    "team_name_tm",
    "team_slug_tm",
    "team_id_tm",
    "season_id",
    "squad_size_with_values",
    "squad_market_value_millions_eur",
    "avg_player_value_millions_eur",
    "url",
]

missing = [col for col in required_cols if col not in tm_raw.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

tm = tm_raw[required_cols].copy()

tm["season_id"] = tm["season_id"].astype(int)

numeric_cols = [
    "squad_size_with_values",
    "squad_market_value_millions_eur",
    "avg_player_value_millions_eur",
]

for col in numeric_cols:
    tm[col] = (
        pd.to_numeric(tm[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

tm["has_market_value"] = (
    tm["squad_market_value_millions_eur"] > 0
).astype(int)

tm = tm.sort_values(["team_name_tm", "season_id"]).reset_index(drop=True)

print("Cleaned shape:", tm.shape)
print("Teams:", tm["team_name_tm"].nunique())
print("Years:", tm["season_id"].min(), "to", tm["season_id"].max())

display(tm.head())

Cleaned shape: (5014, 9)
Teams: 218
Years: 2004 to 2026


,team_name_tm,team_slug_tm,team_id_tm,season_id,squad_size_with_values,squad_market_value_millions_eur,avg_player_value_millions_eur,url,has_market_value
0,Afghanistan,afghanistan,3576,2004,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
1,Afghanistan,afghanistan,3576,2005,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
2,Afghanistan,afghanistan,3576,2006,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
3,Afghanistan,afghanistan,3576,2007,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
4,Afghanistan,afghanistan,3576,2008,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0


## Yearly Normalization

In [4]:
year_stats = (
    tm[tm["squad_market_value_millions_eur"] > 0]
    .groupby("season_id")["squad_market_value_millions_eur"]
    .agg(
        season_market_value_mean="mean",
        season_market_value_median="median",
        season_market_value_std="std",
        season_market_value_min="min",
        season_market_value_max="max",
    )
    .reset_index()
)

tm = tm.merge(year_stats, on="season_id", how="left")

tm["log_market_value"] = np.log1p(
    tm["squad_market_value_millions_eur"]
)

tm["market_value_relative_to_year_mean"] = (
    tm["squad_market_value_millions_eur"] /
    tm["season_market_value_mean"]
)

tm["market_value_relative_to_year_median"] = (
    tm["squad_market_value_millions_eur"] /
    tm["season_market_value_median"]
)

tm["market_value_year_zscore"] = (
    (
        tm["squad_market_value_millions_eur"] -
        tm["season_market_value_mean"]
    )
    /
    tm["season_market_value_std"].replace(0, np.nan)
)

tm["log_market_value_year_centered"] = (
    tm["log_market_value"] -
    tm.groupby("season_id")["log_market_value"].transform("mean")
)

tm["market_value_year_minmax"] = (
    (
        tm["squad_market_value_millions_eur"] -
        tm["season_market_value_min"]
    )
    /
    (
        tm["season_market_value_max"] -
        tm["season_market_value_min"]
    ).replace(0, np.nan)
)

normalized_cols = [
    "season_market_value_mean",
    "season_market_value_median",
    "season_market_value_std",
    "season_market_value_min",
    "season_market_value_max",
    "log_market_value",
    "market_value_relative_to_year_mean",
    "market_value_relative_to_year_median",
    "market_value_year_zscore",
    "log_market_value_year_centered",
    "market_value_year_minmax",
]

for col in normalized_cols:
    tm[col] = tm[col].replace([np.inf, -np.inf], np.nan).fillna(0)

display(tm.head())
display(tm[normalized_cols].describe())

,team_name_tm,team_slug_tm,team_id_tm,season_id,squad_size_with_values,squad_market_value_millions_eur,avg_player_value_millions_eur,url,has_market_value,season_market_value_mean,season_market_value_median,season_market_value_std,season_market_value_min,season_market_value_max,log_market_value,market_value_relative_to_year_mean,market_value_relative_to_year_median,market_value_year_zscore,log_market_value_year_centered,market_value_year_minmax
0,Afghanistan,afghanistan,3576,2004,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,61.036512,19.87,109.598000,0.05,551.70,0.0,0.0,0.0,-0.556913,-1.101439,-0.000091
1,Afghanistan,afghanistan,3576,2005,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,56.016893,19.35,89.187581,0.05,413.15,0.0,0.0,0.0,-0.628080,-1.368667,-0.000121
2,Afghanistan,afghanistan,3576,2006,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,60.508624,24.15,100.570311,0.05,550.50,0.0,0.0,0.0,-0.601655,-1.469805,-0.000091
3,Afghanistan,afghanistan,3576,2007,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,48.983309,12.13,91.877346,0.05,580.50,0.0,0.0,0.0,-0.533138,-1.621006,-0.000086
4,Afghanistan,afghanistan,3576,2008,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,62.683766,14.55,113.717163,0.03,636.90,0.0,0.0,0.0,-0.551225,-1.938522,-0.000047


,season_market_value_mean,season_market_value_median,season_market_value_std,season_market_value_min,season_market_value_max,log_market_value,market_value_relative_to_year_mean,market_value_relative_to_year_median,market_value_year_zscore,log_market_value_year_centered,market_value_year_minmax
count,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5.014000e+03,5014.000000
mean,89.338044,16.093043,183.013393,0.031739,1095.413043,2.175010,0.757479,4.627413,-0.127823,-2.267388e-17,0.063711
std,33.196640,5.774926,76.597238,0.014343,500.687447,2.076396,1.827690,12.058896,0.897110,2.020504e+00,0.152756
min,48.983309,8.880000,89.187581,0.010000,413.150000,0.000000,0.000000,0.000000,-0.628080,-2.881693e+00,-0.000121
25%,61.036512,12.080000,113.717163,0.030000,636.900000,0.048790,0.000600,0.003003,-0.494439,-1.747158e+00,0.000000
50%,70.429425,15.165000,147.044652,0.030000,879.500000,1.695616,0.050281,0.303521,-0.446962,-5.757636e-01,0.004097
75%,123.331313,19.350000,262.815305,0.050000,1566.500000,3.835195,0.569735,3.076209,-0.215957,1.701390e+00,0.048740
max,157.014632,35.025000,308.552029,0.050000,2080.500000,7.640844,14.963336,129.677152,6.502184,5.213376e+00,1.000000


## Coverage Check

In [5]:
coverage_summary = pd.DataFrame([{
    "rows": len(tm),
    "teams": tm["team_name_tm"].nunique(),
    "first_year": tm["season_id"].min(),
    "last_year": tm["season_id"].max(),
    "market_value_non_zero_share": tm["has_market_value"].mean(),
    "missing_or_zero_market_value_share": 1 - tm["has_market_value"].mean(),
}])

display(coverage_summary)

coverage_by_year = (
    tm.groupby("season_id")["has_market_value"]
    .mean()
    .reset_index(name="market_value_coverage")
)

display(coverage_by_year)

,rows,teams,first_year,last_year,market_value_non_zero_share,missing_or_zero_market_value_share
0,5014,218,2004,2026,0.757479,0.242521


,season_id,market_value_coverage
0,2004,0.394495
1,2005,0.472477
2,2006,0.500000
3,2007,0.637615
4,2008,0.706422
5,2009,0.692661
6,2010,0.761468
7,2011,0.848624
8,2012,0.775229
9,2013,0.752294


## Top Teams Sanity Check

In [6]:
latest_year = tm["season_id"].max()

top_latest = (
    tm[tm["season_id"] == latest_year]
    .sort_values("squad_market_value_millions_eur", ascending=False)
    .head(20)
)

display(top_latest[[
    "team_name_tm",
    "season_id",
    "squad_size_with_values",
    "squad_market_value_millions_eur",
    "avg_player_value_millions_eur",
    "market_value_relative_to_year_mean",
    "market_value_year_zscore",
]])

,team_name_tm,season_id,squad_size_with_values,squad_market_value_millions_eur,avg_player_value_millions_eur,market_value_relative_to_year_mean,market_value_year_zscore
1540,France,2026,26,1473.00,56.65,13.396525,5.837247
1333,England,2026,26,1315.00,50.58,11.959559,5.160612
4231,Spain,2026,26,1271.00,48.88,11.559391,4.972182
1632,Germany,2026,26,1005.50,38.67,9.144742,3.835177
3587,Portugal,2026,26,965.00,37.12,8.776406,3.661736
620,Brazil,2026,26,905.20,34.82,8.232542,3.405643
3104,Netherlands,2026,25,763.00,30.52,6.939273,2.796671
183,Argentina,2026,29,762.50,26.29,6.934725,2.794530
3357,Norway,2026,26,586.50,22.56,5.334054,2.040809
436,Belgium,2026,26,558.20,21.47,5.076674,1.919615


## Save Clean Transfermarkt Dataset

In [7]:
tm.to_csv(OUT_FILE, index=False)

print("Transfermarkt clean + normalized dataset saved")
print("Shape:", tm.shape)
print("Saved to:", OUT_FILE)

print("Columns:")
print(tm.columns.tolist())

Transfermarkt clean + normalized dataset saved
Shape: (5014, 20)
Saved to: /workspaces/World_cup_predictor/data/processed/transfermarkt_market_values_clean.csv
Columns:
['team_name_tm', 'team_slug_tm', 'team_id_tm', 'season_id', 'squad_size_with_values', 'squad_market_value_millions_eur', 'avg_player_value_millions_eur', 'url', 'has_market_value', 'season_market_value_mean', 'season_market_value_median', 'season_market_value_std', 'season_market_value_min', 'season_market_value_max', 'log_market_value', 'market_value_relative_to_year_mean', 'market_value_relative_to_year_median', 'market_value_year_zscore', 'log_market_value_year_centered', 'market_value_year_minmax']


## Summary

This notebook creates the clean Transfermarkt team-year dataset.

Main output:

`data/processed/transfermarkt_market_values_clean.csv`

This file is later used by `05_build_model_dataset.ipynb` to join market value features into the final match-level modeling dataset.